# AI-supported descriptive statistics practice

## Purpose of this notebook

In this notebook, you will use AI as a helper while you practise descriptive
statistics on the student unit-attempt extract.

The goal is not to let AI do the work for you. The goal is to use AI to plan an
analysis, draft one piece of code, and explain a result — while **you** check
every claim against numbers you have calculated yourself.

You must keep the code beginner-friendly:

- use `pd.read_excel()` and the methods from this session's notebooks
- use `.mean()`, `.median()`, `.mode()`, `.describe()`, and `.quantile()`
- use `groupby()`, `agg()`, and `sort_values()`
- do not use regular expressions
- do not use libraries this unit has not introduced
- never report an AI number you have not reproduced yourself

## Your task

A course coordinator asks:

> "Which units had unusual mark patterns in Term 1, and should any of them be
> looked at more closely?"

Use `data/Y1_T1_2025.xlsx` (the `Extract` sheet). Your job is to answer with
grouped summaries and outlier fences, using AI as a helper at three points:
planning the analysis, drafting one piece of code, and explaining one result.

Remember the Session 3 rule: a flagged mark is **not automatically wrong** — it
is a record to investigate.

## How to use AI for this task

Use AI for guidance, not blind copying.

A useful prompt is:

> I am a first-year data analytics student learning pandas. I have a DataFrame
> of student unit attempts with the columns `unit_code`, `mark`, `grade_code`,
> `program_name`, and `location_name`. I know `.mean()`, `.median()`,
> `.describe()`, `.quantile()`, `groupby()`, `agg()`, and `sort_values()`.
> Please help me [write your question here]. Keep the code beginner-friendly,
> use only the methods I listed, and explain each step.

After AI suggests code or an explanation, check it:

- Does the code use only columns that exist in the data?
- Does it use only methods you know from this session?
- Does it check the data (missing marks, group sizes) before summarising?
- Does it avoid regular expressions and unfamiliar libraries?
- Can you reproduce every number it mentions?

## Step 1: load and inspect the data first

Before asking AI anything, know your data. AI has not seen this file — you
have.

In [ ]:
import pandas as pd

df_results = pd.read_excel("data/Y1_T1_2025.xlsx", sheet_name="Extract")

print("Rows and columns:", df_results.shape)
print("Missing marks:", df_results["mark"].isna().sum())
df_results.head()

## Step 2: calculate your own baseline

Calculate a grouped summary yourself first. Whatever AI produces later, this
table is what you check it against.

In [ ]:
df_unit_summary = (
    df_results
    .groupby("unit_code")
    .agg(
        attempts=("mark", "size"),
        mean_mark=("mark", "mean"),
        median_mark=("mark", "median"),
        std_mark=("mark", "std"),
    )
    .sort_values("mean_mark")
)

df_unit_summary.round(1).head(10)

## Step 3: ask AI for an analysis plan

Ask AI how it would answer the coordinator's question with your columns and
methods. Then compare its plan with what you know from this session:

- Does the plan inspect the data before summarising?
- Does it report group sizes next to group means?
- Does it use position measures (quartiles, IQR fences) to find unusual marks?

Paste AI's plan into the cell below, and mark each step you agree or disagree
with.

*Paste AI's plan here, with your notes on each step.*

## Step 4: ask AI to draft the outlier fences

Ask AI to write code that:

1. calculates Q1, Q3, and the IQR for `mark`
2. builds the lower fence `Q1 - 1.5 × IQR` and the upper fence
   `Q3 + 1.5 × IQR`
3. counts, for each `unit_code`, how many marks fall outside the fences

Check the draft against the checklist in **How to use AI for this task**, fix
anything that fails the checklist, then paste it below and run it.

In [ ]:
# Paste the checked AI code here, then run it.

## Step 5: verify the result yourself

Now calculate the same thing independently. If your numbers and the AI-drafted
numbers differ, work out which is right before going on — do not assume the
mistake is yours.

In [ ]:
q1 = df_results["mark"].quantile(0.25)
q3 = df_results["mark"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print("Q1:", q1, " Q3:", q3, " IQR:", iqr)
print("Fences:", lower_fence, "to", upper_fence)

df_outside = df_results[
    (results["mark"] < lower_fence) | (results["mark"] > upper_fence)
]

flagged_by_unit = (
    df_outside
    .groupby("unit_code")
    .size()
    .sort_values(ascending=False)
)

print()
print("Flagged marks by unit:")
print(flagged_by_unit.head(10))

## Step 6: ask AI to explain one result — then fact-check it

Pick one unit near the top of your flagged list. Ask AI:

> A unit has a mean mark of [value] but [count] marks flagged outside the IQR
> fences. What could explain this pattern, and what should an analyst check
> before reporting it?

Read the answer critically:

- Which suggested explanations could you actually check in this data?
- Does AI remind you that a flagged mark is not automatically an error?
- Does anything in the answer contradict your numbers?

Use the cell below to look at the flagged records for your chosen unit.

In [ ]:
chosen_unit = flagged_by_unit.index[0]

df_flagged_records = df_outside[df_outside["unit_code"] == chosen_unit]
print("Unit:", chosen_unit)
print("Flagged records:", len(df_flagged_records))
df_flagged_records[["unit_code", "mark", "grade_code", "unit_attempt_status"]].head(10)

## Reflection

Answer these questions in a markdown cell below:

1. Where did AI's plan differ from yours, and whose was better?
2. Did the AI-drafted fence code pass your checklist first time? What did you
   change?
3. Did your verification numbers match the numbers from the AI-drafted code?
4. Which part of AI's explanation could you check in the data, and which part
   was speculation?
5. What would you tell the coordinator, in two sentences?

## What you practised

You practised using AI deliberately at three points of an analysis — planning,
drafting, and explaining — while keeping the analyst's responsibilities:
knowing the data first, verifying every number, and treating flagged values as
questions rather than answers.